This notebook creates all plots regarding the slopiness analysis, allowing to estimate the influence of each parameter in all 3 candidate models. This notebooks contains the code to generate the plots from Figure EV2.

In [1]:
from sympy import *
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib import use
use('Agg')
import warnings
warnings.filterwarnings('ignore')
plt.rc('xtick', labelsize=14)
plt.rc('ytick', labelsize=14)

In [2]:
df_bind = pd.read_csv('data/binding/IL10_data_param.csv')
model_list = ["IL10","IL10_RAp","IL10_MS1"]
directory_plots = "figures/slopiness/"
param_vec_dict = {
    "IL10": [col for col in df_bind.columns[2:-2] if "_f" not in col and "_M_" not in col],
    "IL10_RAp": [col for col in df_bind.columns[2:-1 ] if "_M_" not in col],
    "IL10_MS1": [col for col in df_bind.columns[2:] if "PHOS" not in col]
}
param_vec_plots_dict = {
    "IL10": ['$S_{cell}$','$V_{cell}$','$K_{ILM-b}$','$K_{IL-RA-b}$','$K_{IL-RB-b}$','$K_{IL-RA-RB-b}$','$K_{STAT1-b}$','$K_{STAT3-b}$','$K_{PHOS}$','$K_{DEPHOS}$'],
    "IL10_RAp": ['$S_{cell}$','$V_{cell}$','$K_{ILM-f}$','$K_{ILM-b}$','$K_{IL-RA-f}$','$K_{IL-RA-b}$','$K_{IL-RB-f}$','$K_{IL-RB-b}$','$K_{IL-RA-RB-f}$','$K_{IL-RA-RB-b}$','$K_{STAT1-f}$','$K_{STAT1-b}$','$K_{STAT3-f}$','$K_{STAT3-b}$','$K_{PHOS}$','$K_{DEPHOS}$','$K_{DEPHOS-R}$'],
    "IL10_MS1": ['$S_{cell}$','$V_{cell}$','$K_{ILM-f}$','$K_{ILM-b}$','$K_{IL-RA-f}$','$K_{IL-RA-b}$','$K_{IL-RB-f}$','$K_{IL-RB-b}$','$K_{IL-RA-RB-f}$','$K_{IL-RA-RB-b}$','$K_{STAT1-f}$','$K_{STAT1-b}$','$K_{STAT3-f}$','$K_{STAT3-b}$','$K_{M-f}$','$K_{M-b}$','$M_{0}$']
}

In [3]:
for model in model_list:
    param_vec = param_vec_dict[model]
    param_vec_plots = param_vec_plots_dict[model]
    H = np.loadtxt('results/slopiness/Hessian_nofit_'+model+'.csv', delimiter=',')
    df = pd.DataFrame(H, columns = param_vec, index = param_vec)
    # Plot the second derivative of each parameter to see which has more influence on the cost by itself
    Hii = [df.loc[param,param] for param in param_vec]
    fig,ax=plt.subplots(1,1,figsize=(8, 8), dpi=400)
    plt.bar(param_vec_plots, Hii, width=0.7, color="darkred")
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    plt.ylabel('$H_{i,i}$', fontsize=25)
    plt.yscale('log')
    plt.xticks(rotation=45, ha="right")
    plt.savefig(directory_plots+'/slopiness_diag_'+model+'.png', bbox_inches='tight', transparent=True)
    plt.show()

    # Perform the eigendecomposition of the Hessian
    eigenvalues, eigenvectors = np.linalg.eig(H)
    
    # Sort eigenvalues from big to small
    eigenvalues_sorted = np.sort(eigenvalues,)[::-1]
    eigenvectors_sorted = []
    for eigv in eigenvalues_sorted:
        i = 0
        for eigv2 in eigenvalues:
            if eigv == eigv2:
                break
            i += 1
        eigenvectors_sorted.append(eigenvectors[:, i]/np.linalg.norm(eigenvectors[i]))
    
    # Using symbolic math get the strongest eigenparameters
    param_symbols = []
    for param in param_vec:
        param_symbols.append(symbols(param))
        
    # Get formula of the eigenvectors
    eigenparameters = []
    for vector in eigenvectors_sorted:
        eigenparam_components = []
        i = 0
        for param in param_symbols:
            if abs(vector[i]) >= 0.1:
                eigenparam_components.append(param_symbols[i]**round(vector[i], 3))
            i += 1
        eigenparam = eigenparam_components[0]
        i = 1
        for eigenparam_component in eigenparam_components[1:]:
            eigenparam = eigenparam*eigenparam_components[i]
            i += 1
        eigenparameters.append(eigenparam)

    # Plot the top3 eigenvectors with highest eigenvalue
    fig,ax=plt.subplots(1,1,figsize=(8, 8), dpi=400)
    param_vec = df.columns
    plt.bar(param_vec_plots, eigenvectors_sorted[0]*eigenvalues_sorted[0], width=0.9, color="darkred", alpha=0.8, label="Eigenvalue 1")
    plt.bar(param_vec_plots, eigenvectors_sorted[1]*eigenvalues_sorted[1], width=0.9, color="darkblue", alpha=0.8, label="Eigenvalue 2")
    plt.bar(param_vec_plots, eigenvectors_sorted[2]*eigenvalues_sorted[2], width=0.9, color="darkgreen", alpha=0.8, label="Eigenvalue 3")
    ax.spines["bottom"].set_linewidth(4)
    ax.spines["left"].set_linewidth(4)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.set_tick_params(width=5, length=10)
    ax.xaxis.set_tick_params(width=5, length=10)
    ax.tick_params(axis='x', labelsize=18)
    ax.tick_params(axis='y', labelsize=18)
    plt.ylabel('$\lambda \cdot v$', fontsize=25)
    plt.xticks(rotation=45, ha="right")
    ax.legend(loc="best", fontsize=15)
    plt.savefig(directory_plots+'/slopiness_eigendirections_'+model+'.png', bbox_inches='tight', transparent=True)
    plt.show()